# Byte Pair Encoding 




A language model receives numbers, not raw text. A tokenizer converts text into a sequence of integer **token IDs**.

There are several possible starting points:

| Strategy | Example for `lowest` | Main problem |
|---|---|---|
| Word-level | `[lowest]` | The vocabulary becomes enormous and unseen words need an unknown token |
| Character-level | `[l, o, w, e, s, t]` | Sequences become long and individual tokens carry little meaning |
| Byte-level | UTF-8 byte values | Can represent all text, but sequences initially become even longer |
| Subword | `[low, est]` | Balances vocabulary size and sequence length |

BPE learns useful subword units from data. Frequent patterns become single tokens, while uncommon words can still be represented using smaller pieces.



BPE repeatedly performs the same operation:

1. Start with a sequence of small tokens.
2. Count every adjacent token pair.
3. Find the most frequent pair.
4. Replace every non-overlapping occurrence of that pair with a new token.
5. Add the new token to the vocabulary.
6. Repeat until the desired vocabulary size is reached.

```text
small starting units
        ↓
count adjacent pairs
        ↓
merge the most frequent pair
        ↓
a new, larger token is created
        ↓
repeat
```

BPE does not begin with knowledge of words, prefixes, or grammar. It discovers reusable sequences only from frequency.

Consider the sequence:

```text
b a n a n a
```

Its adjacent pairs are:

```text
(b, a)
(a, n)
(n, a)
(a, n)
(n, a)
```

The frequencies are:

```text
(b, a) → 1
(a, n) → 2
(n, a) → 2
```

There is a tie. A real implementation needs a deterministic tie-breaking rule. If we choose `(a, n)`, create the new token `an`, and merge from left to right, the sequence becomes:

```text
b an an a
```


In [1]:
tokens = ["b", "a", "n", "a", "n", "a"]

pairs = list(zip(tokens, tokens[1:]))
pairs

[('b', 'a'), ('a', 'n'), ('n', 'a'), ('a', 'n'), ('n', 'a')]

Our real tokenizer will represent every token using an integer ID. The pair-counting function should accept a list such as:

```python
[98, 97, 110, 97, 110, 97]
```

and return:

```python
{
    (98, 97): 1,
    (97, 110): 2,
    (110, 97): 2,
}
```

Only adjacent pairs count. For a sequence of length `n`, there are `n - 1` adjacent pair positions.

In [5]:
def count_pairs(token_ids):
    """Return the frequency of each adjacent token-ID pair.

    Example:
        [1, 2, 1, 2] -> {(1, 2): 2, (2, 1): 1}
    """    
    pairs = list(zip(token_ids, token_ids[1:]))
    pair_counts = {}
    for pair in pairs:
        if pair in pair_counts:
            pair_counts[pair] += 1
        else:
            pair_counts[pair] = 1

    return pair_counts

In [7]:
assert count_pairs([]) == {}
assert count_pairs([1]) == {}
assert count_pairs([1, 2, 1, 2]) == {
    (1, 2): 2,
    (2, 1): 1,
}

Merge one pair

Suppose the pair `(97, 110)` receives the new token ID `256`.

```text
Before: [98, 97, 110, 97, 110, 97]
Pair:   (97, 110)
New ID: 256
After:  [98, 256, 256, 97]
```

Scan from left to right:

- if the current token and next token match the target pair, append the new ID and advance by two positions;
- otherwise, append the current token and advance by one position.

This produces non-overlapping merges.

In [22]:
def merge_pair(token_ids, pair, new_token_id):
    """Replace non-overlapping occurrences of pair with new_token_id."""    
    merged_ids = []
    
    i = 0

    while i < len(token_ids):        
        if (
            i < len(token_ids) - 1
            and (token_ids[i], token_ids[i + 1]) == pair
        ):
            merged_ids.append(new_token_id)
            i += 2
        else:
            merged_ids.append(token_ids[i])
            i += 1

    return merged_ids

In [ ]:
assert merge_pair([98, 97, 110, 97, 110, 97], (97, 110), 256) == [98, 256, 256, 97]
assert merge_pair([1, 1, 1, 1], (1, 1), 2) == [2, 2]
assert merge_pair([1], (1, 1), 2) == [1]
print("merge_pair tests passed")

merge_pair tests passed


A byte-level tokenizer begins with a base vocabulary containing all 256 possible byte values:

```text
token IDs 0–255 ↔ byte values 0–255
```

Every learned merge receives a new ID beginning at `256`.

```text
0–255   original byte tokens
256     first learned merge
257     second learned merge
258     third learned merge
...
```

Because any Unicode text can be converted to UTF-8 bytes, this starting vocabulary can represent any valid UTF-8 text.

In [25]:
training_text = "banana banana bandana 🍌"

training_bytes = training_text.encode("utf-8")
training_ids = list(training_bytes)

print("Text:", training_text)
print("Bytes:", training_bytes)
print("Starting token IDs:", training_ids)
print("Number of starting tokens:", len(training_ids))

Text: banana banana bandana 🍌
Bytes: b'banana banana bandana \xf0\x9f\x8d\x8c'
Starting token IDs: [98, 97, 110, 97, 110, 97, 32, 98, 97, 110, 97, 110, 97, 32, 98, 97, 110, 100, 97, 110, 97, 32, 240, 159, 141, 140]
Number of starting tokens: 26


One BPE training step is now:

```text
count pairs
    → choose the most frequent pair
    → assign the next available token ID
    → merge the pair throughout the training sequence
    → record the merge rule
```

For reproducibility, decide how ties will be broken. One simple rule is to use the first pair encountered during the left-to-right scan.

In [26]:
pair_counts = count_pairs(training_ids)
best_pair = max(pair_counts, key=pair_counts.get)
new_token_id = 256
ids_after_one_merge = merge_pair(
    training_ids,
    best_pair,
    new_token_id,
)

print("Most frequent pair:", best_pair)
print("Frequency:", pair_counts[best_pair])
print("New token ID:", new_token_id)
print("Length before:", len(training_ids))
print("Length after: ", len(ids_after_one_merge))

Most frequent pair: (97, 110)
Frequency: 6
New token ID: 256
Length before: 26
Length after:  20


Train multiple merges

Training learns a sequence of merge rules. The order matters:

```text
(token_a, token_b) → 256
(256, token_c)     → 257
(token_d, 257)     → 258
```

A later merge can depend on a token created by an earlier merge.

If the target vocabulary size is `300`, a byte-level tokenizer starts with `256` base tokens and learns `44` merges.

Stop early if the sequence has fewer than two tokens or no adjacent pairs remain.

In [ ]:
def train_bpe(text, vocabulary_size):
    """Learn byte-level BPE merges from text.

    Returns:
        merges: dict mapping (left_id, right_id) to new_token_id
        vocabulary: dict mapping token_id to the bytes represented by it
    """
    if vocabulary_size < 256:
        raise ValueError("A byte-level vocabulary needs at least 256 tokens")

    token_ids = list(text.encode("utf-8"))
    merges = {}
    vocabulary = {token_id: bytes([token_id]) for token_id in range(256)}

    # TODO:
    # 1. Repeat once for each requested merge.
    # 2. Count adjacent pairs.
    # 3. Stop if there are no pairs.
    # 4. Select the most frequent pair.
    # 5. Assign the next token ID.
    # 6. Merge the training sequence.
    # 7. Save the merge rule.
    # 8. Build the new token's bytes from its two children.

    return merges, vocabulary

## 9. Inspect what the tokenizer learned

The vocabulary maps each token ID to the byte sequence it represents. Decode each learned byte sequence only for inspection.

Some individual learned tokens may not be valid UTF-8 by themselves. Use `errors="replace"` when displaying them so inspection does not crash.

In [ ]:
# Run after implementing train_bpe().

# merges, vocabulary = train_bpe(training_text, vocabulary_size=270)

# for pair, token_id in merges.items():
#     piece = vocabulary[token_id].decode("utf-8", errors="replace")
#     print(f"{pair} -> {token_id}: {vocabulary[token_id]!r} {piece!r}")

## 10. Encode new text

Training and encoding are different operations:

- **Training** discovers and records merge rules from a corpus.
- **Encoding** applies those already learned rules to new text without changing them.

Encoding begins with UTF-8 byte IDs, then applies learned merges in training order.

In [ ]:
def encode(text, merges):
    """Encode text using previously learned BPE merge rules."""
    token_ids = list(text.encode("utf-8"))

    # TODO: apply every learned pair in merge order.
    # Python dictionaries preserve insertion order.

    return token_ids

## 11. Decode token IDs

Every vocabulary entry stores the bytes represented by that token. To decode:

1. look up the bytes for each token ID;
2. concatenate all byte sequences;
3. decode the final byte sequence as UTF-8.

Do not decode each token separately. A Unicode character's UTF-8 bytes can be split across several tokens, and those individual pieces may be invalid UTF-8.

In [ ]:
def decode(token_ids, vocabulary):
    """Decode BPE token IDs back into Unicode text."""
    # TODO: concatenate token bytes, then decode once as UTF-8.
    text = ""

    return text

## 12. End-to-end verification

The most important correctness property is a round trip:

```text
decode(encode(text)) == text
```

Test ASCII, accented text, non-Latin scripts, emoji, empty text, and text that was not present in the training corpus.

In [ ]:
# Run after completing all functions.

# merges, vocabulary = train_bpe(training_text, vocabulary_size=270)

# test_strings = [
#     "",
#     "banana",
#     "bandana",
#     "café",
#     "こんにちは",
#     "🍌🐍",
#     "text never seen during training",
# ]

# for text in test_strings:
#     token_ids = encode(text, merges)
#     reconstructed = decode(token_ids, vocabulary)
#     print(repr(text), "->", token_ids, "->", repr(reconstructed))
#     assert reconstructed == text

# print("All round-trip tests passed")

## 13. Measure compression

For this notebook, define a simple compression ratio:

```text
compression ratio = number of UTF-8 bytes / number of BPE tokens
```

A value greater than `1` means the learned merges shortened the token sequence. This is not file compression; it is only a useful way to compare tokenizer sequence lengths.

In [ ]:
def compression_ratio(text, token_ids):
    if not token_ids:
        return 1.0 if not text else float("inf")

    return len(text.encode("utf-8")) / len(token_ids)


# Compare several vocabulary sizes after train_bpe() is complete.
# for size in [256, 260, 270]:
#     merges, vocabulary = train_bpe(training_text, size)
#     ids = encode(training_text, merges)
#     print(size, len(ids), compression_ratio(training_text, ids))

## 14. Important simplifications in our implementation

This notebook teaches the core BPE mechanism. Production tokenizers include additional decisions:

- text normalization;
- pre-tokenization and rules about whether merges may cross boundaries;
- special tokens;
- deterministic tie-breaking;
- reserved vocabulary ranges;
- efficient data structures for training on large corpora;
- a defined policy for invalid UTF-8 during decoding.

Our first implementation treats the entire training string as one byte sequence. This means it may learn merges that include spaces or cross what humans consider word boundaries. That behavior is acceptable for understanding the algorithm, but it must be an explicit design choice in a real tokenizer.

## 15. Questions to answer without notes

- Why does byte-level BPE begin with 256 base tokens?
- What exactly is counted during BPE training?
- Why does every merge reduce sequence length?
- Why does merge order matter?
- What is the difference between tokenizer training and encoding?
- Why must decoding concatenate bytes before calling UTF-8 decode?
- How can byte-level BPE encode a character absent from its training corpus?
- What trade-off changes when vocabulary size increases?
- Why might a production tokenizer pre-tokenize text before applying BPE?

## Completion checklist

- [ ] I implemented `count_pairs`.
- [ ] I implemented `merge_pair`.
- [ ] I implemented `train_bpe`.
- [ ] I implemented `encode`.
- [ ] I implemented `decode`.
- [ ] All round-trip tests pass.
- [ ] I inspected the learned tokens.
- [ ] I compared at least three vocabulary sizes.
- [ ] I can trace one merge by hand.
- [ ] I can explain the complete algorithm without notes.